## 1. PLM
PLM(Pre-trained Language Model)은 대량의 텍스트 데이터를 사전 학습하여 자연어 이해와 생성 능력을 갖춘 인공지능 모델입니다. 

대표적으로 BERT, GPT, T5 등이 있으며, 이들은 대규모 데이터에서 단어의 의미와 문맥을 학습한 후, 특정 작업(예: 문장 분류, 번역, 질의응답 등)에 맞게 추가 학습(Fine-tuning)하여 활용됩니다. 

PLM은 문맥을 고려한 자연어 처리 능력이 뛰어나며, 다양한 언어 기반 AI 애플리케이션에서 핵심 기술로 사용됩니다.

- 대규모 텍스트로 미리 학습한 PLM을 가져온 뒤, 원하는 문제에 맞게 추가학습 하거나 그대로 활용

### Pre-training(사전학습)
대규모 데이터에서 언어의 일반적인 패턴을 먼저 학습하는 단계

### Fine-tuning(미세조정)
이미 학습된 모델을 특정 문젱 맞게 조금 더 학습하는 단계

### Transformer
- Encoder 중심: 보통 BERT 계열
  - 입력 문장 > Transformer Encoder > 문맥을 반영한 표현 > 분류 / 유사도 / 질의응답 등
  - BERT는 문장의 앞뒤 문맥을 함께 참고하는 Encoder 기반 모델
- Decoder 중심: 보통 GPT 계열
  - 이전 Token들 > Transformer Decoder 계열 > 다음 Token 예측 반복
  - GPT 계열은 이전 Token을 바탕으로 다음 Token을 순차적으로 생성하는 Autoregressive 생성 모델
- Encoder-Decoder 중심: T5 계열
  - Encoder과 Decoder을 모두 사용해서 여러 NLP 문제를 Text-to-Text 형태로 통일해서 처리하는 대표적인 모델

# BERT
- BERT의 대표적인 사전학습 목표는 MLM(Masked Language Modeling)
```
대한민국의 수도는 [MASK] 입니다.
```
> `모델은 주변 문맥을 보고 [MASK] 위치의 단어를 맞히도록 학습`

```
대한민국의 수도는 [MASK] 입니다. > Transformer Encoder > "서울"
```

- 위 과정을 엄청나게 많은 문장에서 반복하면 모델 내부에는 단순한 단어 사전이 아니라 문맥에 따라
달라지는 표현이 만들어지게 됨

```
배를 먹었다.        -> 과일
배가 바다에 있다.   -> 선박
배가 아프다.        -> 신체
```


In [ ]:
import pandas as pd
import numpy as np
import random
import time
import datetime
import csv
import os
import torch
import torch.nn.functional as F

from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from datasets import load_dataset
from torch.optim import AdamW
from transformers import pipeline
from transformers import BertTokenizer
from transformers import AutoModelForSequenceClassification   #, BertConfig
from transformers import AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss

In [2]:
from pathlib import Path
import subprocess

In [3]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
REPO_DIR = DATA_DIR / "K-MHaS"

if not REPO_DIR.exists():
  subprocess.run(["git", "clone", "https://github.com/adlnlp/K-MHaS", str(REPO_DIR)], check=True)
else: 
  print(f"이미 저장소가 존재합니다.\n{REPO_DIR}")

DATA_DIR = REPO_DIR / "data"

PROJECT_DIR, DATA_DIR, REPO_DIR

이미 저장소가 존재합니다.
c:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_09_11_NLP\data\K-MHaS


(WindowsPath('c:/LANG_CHAIN_2026/2026-05-19_KDT_lang_chain/01_class/2026_09_11_NLP'),
 WindowsPath('c:/LANG_CHAIN_2026/2026-05-19_KDT_lang_chain/01_class/2026_09_11_NLP/data/K-MHaS/data'),
 WindowsPath('c:/LANG_CHAIN_2026/2026-05-19_KDT_lang_chain/01_class/2026_09_11_NLP/data/K-MHaS'))

In [4]:
for p in DATA_DIR.iterdir():
  print(p.name)

kmhas_test.txt
kmhas_train.txt
kmhas_valid.txt


In [5]:
trian_file = next(DATA_DIR.rglob("*_train.txt"))
trian_file

WindowsPath('c:/LANG_CHAIN_2026/2026-05-19_KDT_lang_chain/01_class/2026_09_11_NLP/data/K-MHaS/data/kmhas_train.txt')

In [6]:
with trian_file.open("r", encoding="utf-8") as f:
  for _ in range(3):
    print(f.readline().rstrip())

document	label
"자한당틀딱들.. 악플질 고만해라."	2,4
정치적으로 편향된 평론한은 분은 별로...	8


# **3. K-MHaS 데이터셋**
- 한국어 혐오 표현 데이터셋
- https://github.com/adlnlp/K-MHaS.git
- Multi-label classification

In [7]:
# load_dataset("csv", ...)는 CSV/TSV 형태의 데이터를 Dataset 객체로 읽어줌
train = load_dataset(
  "csv", 
  data_files=str(DATA_DIR / "kmhas_train.txt"), 
  delimiter="\t", 
  # split="train" 은 원본 K-MHaS의 train/val/test의 의미를 새로 결정하는 옵션이 아니라 data_files로 파일 하나를 전달했을 때 Datasets가 만든 하나의 split를 바로 반환하기 위함
  split="train"   
)

validation = load_dataset(
  "csv", 
  data_files=str(DATA_DIR / "kmhas_valid.txt"), 
  delimiter="\t", 
  split="train"   
)

test = load_dataset(
  "csv", 
  data_files=str(DATA_DIR / "kmhas_test.txt"), 
  delimiter="\t", 
  split="train"   
)

train, validation, test

(Dataset({
     features: ['document', 'label'],
     num_rows: 78977
 }),
 Dataset({
     features: ['document', 'label'],
     num_rows: 8776
 }),
 Dataset({
     features: ['document', 'label'],
     num_rows: 21939
 }))

In [9]:
train[0]
print("문장:", train[0]["document"])
print("라벨:", train[0]["label"])

문장: 자한당틀딱들.. 악플질 고만해라.
라벨: 2,4


### 4. Multi-label Classification
- Multi-class
```
하나의 입력에 정답이 하나
이미지 > 고양이 / 강아지 / 토끼 중 하나
```
- Multi-label
```
하나의 라벨이 여러 범주에 동시에 해당할 수 있음
문장 > 연령차별=1, 혐오욕설=1, 성차별=0
각 레이블이 서로 독립적이므로 각 출력에 Sigmoid를 적용
logit > sigmoid > 각 label의 확률
```

### [CLS], [SEP], [PAD]
BERT는 특별한 역할을 가진 Special Token이 있음
- [CLS]: 문장 전체를 대표하는 위치로 주로 분류에 사용
- [SEP]: 문장 또는 문장 쌍의 경계를 표시
- [PAD]: 길이가 다른 문장을 같은 길이로 맞출 때 사용
- [MASK]: BERT의 MLM에 사용
- [UNK]: vocabulary에 없는 token을 표현

### Multi-label 정답을 Multi-hot Vector로 변환
- 신경망이 클래스를 학습하기 좋읂 형태는 아래와 같음
- label이 [1, 3]일 때, 클래스가 총 5개라면 신경망이 학습하기 좋은 형태는 [0, 1, 0, 1, 0]
- BultiLabelBinarizer을 사용하여 변환을 편리하게 사용 가능

In [10]:
def parse_label(value):
  return [int(x) for x in str(value).strip().split(",")]

In [11]:
train_y = [parse_label(x) for x in train["label"]]
validation_y = [parse_label(x) for x in validation["label"]]
test_y = [parse_label(x) for x in test["label"]]

train_y[:2], validation_y[:2], test_y[:2]

([[2, 4], [8]], [[8], [8]], [[8], [2, 3]])

In [13]:
# 주의! train/validation/test 각각 fit 하면 안됨
# 같은 encoder로 train / validation / test transform 해야 클래스 번호의 의미도 모든 데이터에서 동일하게 유지
enc = MultiLabelBinarizer()
enc.fit(train_y)
num_labels = len(enc.classes_)
print("클래스:", enc.classes_)
print("클래스 수:", num_labels)
print("원래 label:", train_y[0])

클래스: [0 1 2 3 4 5 6 7 8]
클래스 수: 9
원래 label: [2, 4]


In [15]:
train_labels = torch.tensor(enc.transform(train_y), dtype=torch.float32)
validation_labels = torch.tensor(enc.transform(validation_y), dtype=torch.float32)
test_labels = torch.tensor(enc.transform(test_y), dtype=torch.float32)

In [16]:
train_labels[0]

tensor([0., 0., 1., 0., 1., 0., 0., 0., 0.])

### 5. KLUE-BERT Tokenizer
KLUE = 한글 대규모 copus dataset
- KLUE-BERT는 한국어 데이터로 사전학습된 BERT 계열 모델
- Transformers 코드에서는 특정 클래스인 BertTokenizer 보다 AutoTokenizer을 통해 알아서 사용하는 것이 범용적임

In [ ]:


MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("CLS:", tokenizer.cls_token, tokenizer.cls_token_id)
print("SEP:", tokenizer.sep_token, tokenizer.sep_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)
print("MASK:", tokenizer.mask_token, tokenizer.mask_token_id)

Tokenizer: BertTokenizer
CLS: [CLS] 2
SEP: [SEP] 3
PAD: [PAD] 0
MASK: [MASK] 4


```
Tokenizer
문장 > Subword Tokenization > Token > TokenID > [CLS], [SEP] 등 추가 > Padding / Truncation / attention_mask 생성
````

In [30]:
sample_text = str(train[0]["document"])
sample = tokenizer(sample_text, add_special_tokens=True, truncation=True, max_length=64, return_tensors="pt", padding="max_length")
sample_text, sample["input_ids"], tokenizer.convert_ids_to_tokens(sample["input_ids"][0]), sample["attention_mask"]

('자한당틀딱들.. 악플질 고만해라.',
 tensor([[    2,  1517,  2470,  2481,  2362,  2587,  2031,    18,    18,  1377,
           2546,  2431, 22333,  2097,  2181,    18,     3,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0]]),
 ['[CLS]',
  '자',
  '##한',
  '##당',
  '##틀',
  '##딱',
  '##들',
  '.',
  '.',
  '악',
  '##플',
  '##질',
  '고만',
  '##해',
  '##라',
  '.',
  '[SEP]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  '[PAD]',
  

### input_ids
- Token을 vocabulary의 정수 번호로 바꾼 값
```
[CLS] 나는 학교에 간다 [SEP]
  1   123   456
```

### attention_mask
- 실제 token과 padding를 구분
```
input
[CLS] 나는 [SEP] [PAD] [PAD]
attention mask
  1    1    1     0      0
```

> attention mask는 padding를 중요한 정보처럼 참고하지 않도록 하는 역할

In [ ]:
MAX_LEN = 128

def encode_texts(texts, labels, tokenizer, max_len=MAX_LEN):
  encoded = tokenizer(["" if x is None else str(x) for x in texts],
                      add_special_tokens=True, truncation=True, max_length=max_len, 
                      return_tensors="pt", padding="max_length")
  
  return (encoded["input_ids"], encoded["attention_mask"], labels.to(torch.float32))

In [34]:
train_inputs, train_mask, train_labels = encode_texts(train["document"], train_labels, tokenizer)
validation_inputs, validation_mask, validation_labels = encode_texts(validation["document"], validation_labels, tokenizer)
test_inputs, test_mask, test_labels = encode_texts(test["document"], test_labels, tokenizer)

In [36]:
print("input_ids:", train_inputs.shape)
print("attention_mask:", train_mask.shape)
print("labels:", train_labels.shape)

input_ids: torch.Size([78977, 128])
attention_mask: torch.Size([78977, 128])
labels: torch.Size([78977, 9])


In [37]:
BATCH_SIZE = 32

train_data = TensorDataset(train_inputs, train_mask, train_labels)
validation_data = TensorDataset(validation_inputs, validation_mask, validation_labels)
test_data = TensorDataset(test_inputs, test_mask, test_labels)

train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=BATCH_SIZE)
validation_dataloader = DataLoader(validation_data, sampler=SequentialSampler(validation_data),
                                   batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_data, sampler=SequentialSampler(test_data), batch_size=BATCH_SIZE)

In [38]:
batch = next(iter(train_dataloader))
print(batch[0].shape)
print(batch[1].shape)
print(batch[2].shape)

torch.Size([32, 128])
torch.Size([32, 128])
torch.Size([32, 9])


# 6. KLUE-BERT Multi-label Classification
- 사전 학습 된 BERT에 분류 HEAD를 붙임
- 문장 > Tokenizer > TokenIDs > Pre-trained BERT(Transformer-encoder) > representation > Linear Classification Head > num_labels개의 logits > sigmoid > 각 label의 확률

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

# multi_label_classification - 멀티 라벨 분류
# single_label_classification - 이진 분류
# 모델에 사용하는 레이블이 서로 독립적이므로 각 출력에 Sigmoid를 적용. 학습에서는 보통 BCEWithLogitsLoss 계열 목적함수를 사용
# AutoModelForSequenceClassification에 problem_type="multi_label_classification"을 설정하면 자동으로 Transformers 모델이 유형에 맞는 loss를 설정
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, problem_type="multi_label_classification").to(DEVICE)
model

cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [ ]:
print(type(model).__name__)
print(model.config.num_labels)

BertForSequenceClassification
9


# 7. Fine-tuning
- Pre-trained BERT parameters + Classification Head parameters = 모두 업데이트
- Full Fine-tuning
- 데이터가 작거나 GPU 메모리가 제한적인 경우에는 일부 layer을 frreze하거나 LoRA같은 PEFT(Parameter Efficient Fine-Tuning)

In [53]:
EPOCHS = 10
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_dataloader) * EPOCHS
total_steps

24690

In [48]:
# get_linear_schedule_with_warmup: 허깅페이스 transformers에서 제공하는 함수
# 전체 학습의 10% (num_warmup_steps)를 warmup에 사용한다.
# 전체 학습이 100batch라면 10step동안 lr 올리고 90step동안 lr 내리기
# 이미 parameter 자체가 잘 학습이 되어있기 때문에 갑자기 바꾸지 않고 천천히 변경하게 하기. 
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

print("전체 학습 step:", total_steps)

전체 학습 step: 12345


In [49]:
def multi_label_metrics(logits, labels, threshold=0.5):
    probs = torch.sigmoid(torch.as_tensor(logits)).cpu().numpy()
    y_true = np.asarray(labels)
    y_pred = (probs >= threshold).astype(int)

    metrics = {
        # subset_accuracy: 한 샘플의 모든 라벨이 맞아야함
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        # 전체 라벨중에서 잘못 예측한 label의 비율 (낮을수록 좋음)
        "hamming_loss": hamming_loss(y_true, y_pred),
    }

    try:
        metrics["roc_auc_micro"] = roc_auc_score(
            y_true,
            probs,
            average="micro"
        )
    except ValueError:
        metrics["roc_auc_micro"] = float("nan")

    return metrics

In [50]:
def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round(elapsed))))


def evaluate(model, dataloader, device):
    model.eval()

    all_logits = []
    all_labels = []

    for batch in tqdm(dataloader, leave=False):
        input_ids, attention_mask, labels = [x.to(device) for x in batch]

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    return multi_label_metrics(logits, labels)

In [51]:
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [54]:
for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch + 1} / {EPOCHS} ========")

    start = time.time()
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_dataloader):
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_dataloader)

    print(f"Train Loss: {avg_loss:.4f}")
    print("Time:", format_time(time.time() - start))

    valid_metrics = evaluate(
        model,
        validation_dataloader,
        DEVICE
    )

    for name, value in valid_metrics.items():
        print(f"{name}: {value:.4f}")


======== Epoch 1 / 10 ========


100%|██████████| 2469/2469 [08:05<00:00,  5.09it/s]


Train Loss: 0.1294
Time: 0:08:05


subset_accuracy: 0.7954
f1_micro: 0.8444
f1_macro: 0.7106
f1_weighted: 0.8407
hamming_loss: 0.0382
roc_auc_micro: 0.9832

======== Epoch 2 / 10 ========


100%|██████████| 2469/2469 [08:08<00:00,  5.05it/s]


...

Train Loss: 0.0152
Time: 0:08:13


subset_accuracy: 0.8009
f1_micro: 0.8463
f1_macro: 0.7500
f1_weighted: 0.8466
hamming_loss: 0.0388
roc_auc_micro: 0.9728

======== Epoch 10 / 10 ========


100%|██████████| 2469/2469 [08:13<00:00,  5.01it/s]


Train Loss: 0.0126
Time: 0:08:13


subset_accuracy: 0.8003
f1_micro: 0.8467
f1_macro: 0.7528
f1_weighted: 0.8469
hamming_loss: 0.0385
roc_auc_micro: 0.9736


In [55]:
MODEL_DIR = Path.cwd() / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "KLUE_BERT_multilabel.pt"
torch.save(model.state_dict(), MODEL_PATH)
print("저장 완료:", MODEL_PATH.resolve())

저장 완료: C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_09_11_NLP\model\KLUE_BERT_multilabel.pt


In [58]:
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, problem_type="multi_label_classification"
)

state_dict = torch.load(MODEL_PATH, map_location="cpu")
loaded_model.load_state_dict(state_dict)
loaded_model.to(DEVICE)
loaded_model.eval()
print("모델 로드 완료")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 로드 완료


In [59]:
test_metrics = evaluate(loaded_model, test_dataloader, DEVICE)

In [60]:
test_metrics

{'subset_accuracy': 0.7947946579151283,
 'f1_micro': 0.8443539370707555,
 'f1_macro': 0.7676644885260747,
 'f1_weighted': 0.8451197073929446,
 'hamming_loss': 0.03948321355678117,
 'roc_auc_micro': 0.974034188337506}

## 8. Hugging Face pipeline
- 기존 과정: 문장 > Tokenizer > Tensor > Model > Logits > Sigmoid > Label
- pipeline()은 위 과정을 편리하게 묶어주는 고수준 추론 API

In [65]:
# 감성 분석: sentiment-analysis
# 질의 응답: question-answering
# 빈칸 채우기: fill-mask
# 텍스트 생성: text-generation
# 요약: summarization
# 번역: translation
pipe = pipeline(
  task="text-classification",
  model=loaded_model,
  tokenizer=tokenizer,
  device=DEVICE,
  truncation=True,
  max_length=MAX_LEN,
  function_to_apply="sigmoid",
  top_k=None
)

In [71]:
print(pipe("바나나"))

[[{'label': 'LABEL_8', 'score': 0.9991047978401184}, {'label': 'LABEL_1', 'score': 0.000278446648735553}, {'label': 'LABEL_3', 'score': 0.0001642579591134563}, {'label': 'LABEL_0', 'score': 0.00012724380940198898}, {'label': 'LABEL_4', 'score': 0.00012401834828779101}, {'label': 'LABEL_5', 'score': 8.921966946218163e-05}, {'label': 'LABEL_2', 'score': 7.641000411240384e-05}, {'label': 'LABEL_6', 'score': 7.239800470415503e-05}, {'label': 'LABEL_7', 'score': 4.421317862579599e-05}]]


In [74]:
label_dict = {
    "LABEL_0": "출신차별",
    "LABEL_1": "외모차별",
    "LABEL_2": "정치성향차별",
    "LABEL_3": "혐오욕설",
    "LABEL_4": "연령차별",
    "LABEL_5": "성차별",
    "LABEL_6": "인종차별",
    "LABEL_7": "종교차별",
    "LABEL_8": "해당사항없음",
}

In [77]:
def prediction(text, threshold=0.5):
    results = pipe(text)
    if results and isinstance(results[0], list):
        results = results[0]

    selected = []

    for item in results:
        if item['score'] >= threshold:
            selected.append({
                "label": label_dict.get(item['label'], item['label']),
                "score": round(float(item["score"]), 4)
            })
    return selected

In [ ]:
prediction("이런 꼰대 아저씨들 같으니")

[{'label': '연령차별', 'score': 0.8425}, {'label': '외모차별', 'score': 0.6349}]

In [ ]:
prediction("니거")  

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'label': '인종차별', 'score': 0.9828}, {'label': '출신차별', 'score': 0.573}]

: 